<img src="../../../At-Home-Round/Radar/figs/IOAI-Logo.png" alt="Logo IOAI" width="200" height="auto">

[IOAI 2025 (Pekin, Chine), manche a domicile](https://ioai-official.org/china-2025)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2025/blob/main/fr/At-Home-Round/Radar/Radar.ipynb)

# Radar

Le radar est une technologie cle des communications sans fil, avec de nombreuses applications comme les voitures autonomes. Il met typiquement en jeu une antenne qui emet un signal precis et recoit ses reflexions sur les objets de l'environnement. En traitant ces signaux, le systeme determine la direction angulaire, la distance et la vitesse des objets cibles.

Dans la pratique, le traitement des signaux radar est rendu difficile par le bruit et les reflexions des objets non cibles. Par exemple, lors de la detection de pietons, le radar peut aussi recevoir des reflexions provenant d'arbres ou d'autres elements d'arriere-plan, ce qui peut degrader la precision. Votre tache est d'utiliser l'IA pour analyser les signaux recus et identifier la presence d'un humain a chaque position.

## Donnees

Pour mesurer les objets autour du radar, on utilise les parametres cles suivants :

- **Range (portee)** : distance en ligne droite entre le radar et un objet.
- **Azimuth** : angle horizontal (gauche-droite) entre le radar et l'objet.
- **Elevation** : angle vertical (haut-bas) de l'objet par rapport au radar.
- **Velocity (vitesse)** : vitesse a laquelle l'objet se rapproche ou s'eloigne du radar.

<img src="../../../At-Home-Round/Radar/figs/Radar Fig 1.png" width="300">

Les donnees radar sont transformees en plusieurs **heatmaps** (cartes thermiques), chacune encodant l'**intensite du signal recu** a differentes positions et directions.
- Les **heatmaps statiques** mettent en evidence les reflexions des objets **immobiles**.
- Les **heatmaps dynamiques** mettent en evidence les changements dus aux objets **en mouvement**.

Lorsqu'aucun objet n'est present a un endroit donne, le signal est principalement du bruit d'arriere-plan et reste faible. Au contraire, les reflexions sur un objet augmentent l'intensite du signal et permettent la detection.

Par exemple, la **heatmap statique range-azimuth** represente l'intensite du signal selon differentes distances (**range**) et angles horizontaux (**azimuth**), principalement issue des reflexions sur les objets fixes.

Chaque echantillon est stocke dans un fichier `.mat.pt` sous forme de tenseur de forme $7 \times 50 \times 181$ ou :
- $7$ correspond au nombre de cartes (6 heatmaps + 1 carte de labels semantiques),
- $50$ aux bacs de distance (range bins),
- $181$ aux bacs angulaires, couvrant les angles de $-90^\circ$ a $+90^\circ$ dans le plan horizontal ou vertical.

Les 6 heatmaps sont structurees ainsi :

- **Index 0** : heatmap statique range-azimuth
- **Index 1** : heatmap dynamique range-azimuth
- **Index 2** : heatmap statique range-elevation
- **Index 3** : heatmap dynamique range-elevation
- **Index 4** : heatmap statique range-velocity
- **Index 5** : heatmap dynamique range-velocity

Toutes les valeurs sont **normalisees** : aucune conversion d'unite n'est requise.

La **carte a l'index 6** est la carte de labels semantiques, en format range-azimuth. Chaque pixel indique si un humain est present a cette position, en valeur binaire :
- **0** : humain absent
- **1** : humain present

Voici un extrait d'un echantillon du jeu de donnees :

<img src="../../../At-Home-Round/Radar/figs/Radar Fig 2.png" width="300">

## Tache

Votre tache est de developper un modele qui prend en entree les **six premieres heatmaps** (indices $0$ a $5$) et predit la **carte de labels semantiques** (index $6$) en sortie. L'objectif est d'identifier precisement si une **cible humaine** est presente (1) ou absente (0) a chaque position du champ de vision du radar.

1. **Entree** : tenseur de forme **$6 \times 50 \times 181$** representant les six heatmaps radar.
2. **Sortie** : tenseur de forme **$50 \times 181$** representant la carte de labels cible.

## Acces aux donnees
Vous pouvez telecharger les ensembles d'entrainement et de validation via le lien ci-dessous :

```
!pip install gdown
```

```
!gdown https://drive.google.com/uc?id=1mXqBIqSfHif3LvJ7Jce1C7addqdfu7B-
!unzip Millimeter-wave_dataset.zip
```

## Visualisation du jeu de donnees
Vous pouvez selectionner une donnee a visualiser pour comprendre concretement ce que represente chaque heatmap.
```python
import torch
import matplotlib.pyplot as plt
import numpy as np
import matplotlib
from matplotlib import rcParams
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

def visualize_pt_file(file_path):
    data = torch.load(file_path)
    print(f"Loaded data shape: {data.shape}")
    num_images = data.shape[0]
    cols = 3
    rows = (num_images + cols - 1) // cols

    fig = plt.figure(figsize=(cols * 5.5, rows * 5.5), constrained_layout=True)

    gs = gridspec.GridSpec(rows, cols, figure=fig)

    colors = ['black', 'yellow']
    cmap_discrete = ListedColormap(colors)

    for i in range(num_images):
        img = data[i].numpy().squeeze()
        img = np.flipud(img)

        if i < 6:
            ax = fig.add_subplot(gs[i // cols, i % cols], projection='polar')
            ...
```
```
file_path = f'/content/training_set/1.mat.pt'
visualize_pt_file(file_path)
```

Loaded data shape: torch.Size([7, 50, 181])

<img src="../../../At-Home-Round/Radar/figs/Radar Fig 3.png" width="300">

## Evaluation
Le score de cette tache repose sur la **precision d'identification des labels**. Identifier correctement les points cibles compte plus que d'identifier correctement les points d'arriere-plan.

Plus precisement :
- Chaque point d'**arriere-plan** correctement identifie rapporte **1 point**.
- Chaque point **cible** correctement identifie rapporte **1500 points**.
- Le score final est normalise sur **0-1** en comparant au score maximal theorique.

La fonction suivante calcule votre score :

```
import torch
def cal_accuracy(model, test_loader, bonus):
    model.eval()
    total_score = 0
    total_theo = 0

    with torch.no_grad():
        for images, labels, _ in test_loader:
            images = images.cuda() if torch.cuda.is_available() else images
            labels = labels.cuda() if torch.cuda.is_available() else labels

            outputs = model(images)
            outputs = torch.argmax(outputs, dim=1)

            equal_mask = outputs == labels  # masques correctement predits
            neg_one_mask = labels == 0      # masque des categories d'arriere-plan

            # Calcule le score
            score_neg_one = (equal_mask & neg_one_mask).sum() * 1  # Score arriere-plan
            score_other = (equal_mask & ~neg_one_mask).sum() * bonus  # Score cible
            score_theo = neg_one_mask.sum() * 1 + (~neg_one_mask).sum() * bonus  # Score theorique maximal

            total_score += score_neg_one + score_other
            total_theo += score_theo

    score = total_score.item() / total_theo.item()
    return score
```

## Exemple
Pour une heatmap $3\times3$, supposons que la verite terrain soit
$$
\begin{bmatrix}
0 & 0 & 0 \\
1 & 1 & 1 \\
0 & 0 & 0
\end{bmatrix}
$$
et que le resultat identifie soit
$$
\begin{bmatrix}
0 & 1 & 0 \\
0 & 1 & 0 \\
0 & 1 & 0
\end{bmatrix}
$$
Il y a alors quatre $0$ correctement identifies et un $1$ correctement identifie. Votre score est $4 + 1500 = 1504$ points. Le score maximal possible est $6 + 1500 \times 3 = 4506$, c'est-a-dire celui de six $0$ et trois $1$. Votre score normalise est $1504 / 4506 = 0.33$.
$$
Score = \frac{4 \times 1 + 1 \times 1500}{6 \times 1 + 3 \times 1500}=0.33
$$



In [ ]:
import numpy as np
import pandas as pd 
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import os
import sys
sys.path.append('/bohr/train-4gug/v2')
from dataloader import load_data

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, padding=1)  
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)  
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=2, kernel_size=3, padding=1)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)  
        return x

def train(model, train_loader, test_loader, optimizer, criterion, num_epochs=100):
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        batch_count = 0
        
        for images, labels, _ in train_loader:
            images = images.cuda() if torch.cuda.is_available() else images
            labels = labels.cuda() if torch.cuda.is_available() else labels
            
            outputs = model(images)
            outputs = outputs.view(outputs.size(0), outputs.size(1), -1)  # [B, C, H*W]
            labels = labels.view(labels.size(0), -1)  # [B, H*W]
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            batch_count += 1
        
        avg_train_loss = epoch_loss / batch_count
        train_losses.append(avg_train_loss)
        
        model.eval()
        val_loss = 0.0
        val_batch_count = 0
        
        with torch.no_grad():
            for images, labels, _ in test_loader:
                images = images.cuda() if torch.cuda.is_available() else images
                labels = labels.cuda() if torch.cuda.is_available() else labels
                
                outputs = model(images)
                outputs = outputs.view(outputs.size(0), outputs.size(1), -1)
                labels = labels.view(labels.size(0), -1)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                val_batch_count += 1
        
        avg_val_loss = val_loss / val_batch_count
        val_losses.append(avg_val_loss)
        
        if (epoch+1) % 2 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], '
                  f'Train Loss: {avg_train_loss:.4f}, '
                  f'Val Loss: {avg_val_loss:.4f}')
    
    return train_losses, val_losses

data_path = '/bohr/train-4gug/v2/training_set'

train_loader, test_loader = load_data(
    base_path=data_path,
    batch_size=4,  
    test_size=0.2
)

model = MyModel()
if torch.cuda.is_available():
    model = model.cuda()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) 

train_losses, val_losses = train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=40
)

In [ ]:
# Ecrivez ci-dessous le code de votre modele (y compris les imports necessaires comme torch et torch.nn) pour generer un fichier de structure de modele facilement chargeable par la plateforme de notation
model_code = """  
import torch
import torch.nn as nn
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=2, kernel_size=3, padding=1)  # 5 categories

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)  
        return x
"""
# Ecrit le code dans un fichier
with open('submission_model.py', 'w',encoding="utf-8") as f:
    f.write(model_code)
print("Le fichier submission_model.py a ete genere.")

In [ ]:
# Sauvegarde les parametres du modele
torch.save(model.state_dict(), 'submission_dic.pth')
print("Le fichier submission_dic.pth a ete sauvegarde.")

In [ ]:
# Ce bloc precise le format de soumission de la question.
import zipfile
import os

# Definit les fichiers a inclure et le nom de l'archive.
files_to_zip = ['submission_model.py', 'submission_dic.pth']
zip_filename = 'submission.zip'

# Cree l'archive zip
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # Ajoute le fichier a l'archive
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} cree avec succes !')